# Course 7: Geometric Deep Learning and Graphs
## Session 9: Generative models on graphs and manifolds

**Goal:** train a small discrete diffusion model that generates graphs with two communities, and check its samples.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Training graphs
Small graphs with two tight communities. The model should learn that structure from examples.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
N = 12
def community_graph(rng, p_in=0.9, p_out=0.05):
    lab = rng.permutation(np.repeat([0, 1], N // 2))
    P = np.where(lab[:, None] == lab[None, :], p_in, p_out)
    A = np.triu(rng.random((N, N)) < P, 1); return (A | A.T).astype(np.float32)

rng = np.random.default_rng(0)
data = torch.tensor(np.stack([community_graph(rng) for _ in range(400)]))

def stats(A):
    """Edge density and clustering (how often two neighbours of a node are themselves connected)."""
    A = A.float(); deg = A.sum(-1)
    tri = torch.diagonal(A @ A @ A, dim1=-2, dim2=-1).sum(-1); wedges = (deg * (deg - 1)).sum(-1).clamp(min=1)
    return A.mean().item(), (tri / wedges).mean().item()

print("training graphs: density %.3f  clustering %.3f" % stats(data))

### 2. The forward process: flip edges at random
At noise level `t`, every edge slot is replaced by a coin flip with probability `t`. At `t = 1` the graph is pure noise.

In [ ]:
def sym(M):
    U = torch.triu(M, 1); return U + U.transpose(-1, -2)

def noise(A, t):
    replace = sym((torch.rand_like(A) < t).float()); coin = sym((torch.rand_like(A) < 0.5).float())
    return A * (1 - replace) + coin * replace

torch.manual_seed(0)
for t in [0.0, 0.3, 0.7, 1.0]:
    print("t = %.1f  density %.3f  clustering %.3f" % ((t,) + stats(noise(data, torch.tensor(t)))))

### 3. The denoiser
For every pair of nodes it predicts whether the clean graph has an edge there. Its inputs are built from the noisy graph in a way that does not depend on node order: the noisy edge, the number of shared neighbours, paths of length 3, and the two degrees.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(6, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 1))
    def forward(self, A, t):                                  # A: (B, N, N)   t: (B,)
        deg = A.sum(-1) / N; A2 = A @ A / N; A3 = A2 @ A / N
        f = torch.stack([A, A2, A3, deg[:, :, None] + deg[:, None, :], (deg[:, :, None] - deg[:, None, :]).abs(),
                         t[:, None, None].expand_as(A)], -1)
        return sym(self.net(f).squeeze(-1))                   # logits for "there is an edge in the clean graph"

torch.manual_seed(0)
model = Denoiser(); opt = torch.optim.Adam(model.parameters(), lr=2e-3)
mask = torch.triu(torch.ones(N, N), 1).bool()
for step in range(4000):
    A0 = data[torch.randint(len(data), (64,))]
    t = torch.rand(64)
    At = noise(A0, t[:, None, None])
    loss = F.binary_cross_entropy_with_logits(model(At, t)[:, mask], A0[:, mask])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0 or step == 3999: print(f"step {step:4d}  loss {loss.item():.3f}")

### 4. Sampling: start from noise and denoise step by step
At each step the model guesses the clean graph, and we re-noise that guess to the next, lower, noise level.

In [ ]:
@torch.no_grad()
def sample(n, steps=60):
    A = sym((torch.rand(n, N, N) < 0.5).float())
    for t in torch.linspace(1, 0, steps + 1)[:-1]:
        p = torch.sigmoid(model(A, t.expand(n)))
        last = t <= 1 / steps + 1e-6
        A0_hat = sym((p > 0.5).float()) if last else sym((torch.rand_like(p) < p).float())    # final step: take the most likely graph
        A = noise(A0_hat, (t - 1 / steps).clamp(min=0))
    return A

torch.manual_seed(1)
S = sample(200)
d_data, c_data = stats(data); d_s, c_s = stats(S); d_n, c_n = stats(sym((torch.rand(200, N, N) < d_data).float()))
print("training graphs       : density %.3f  clustering %.3f" % (d_data, c_data))
print("diffusion samples     : density %.3f  clustering %.3f" % (d_s, c_s))
print("random, same density  : density %.3f  clustering %.3f" % (d_n, c_n))
assert abs(c_s - c_data) < abs(c_n - c_data)

The samples sit between random graphs and the training graphs. A model this small, trained for a minute, learns the structure only partly. Closing the gap is one of the TODO items.

### 5. Do the samples have two communities?
The second eigenvector of the Laplacian splits a graph in two. Count how many edges fall inside the two halves.

In [ ]:
def inside_fraction(A):
    out = []
    for a in A.numpy():
        d = a.sum(1); L = np.diag(d) - a
        side = np.linalg.eigh(L)[1][:, 1] > 0
        same = side[:, None] == side[None, :]
        out.append((a * same).sum() / max(a.sum(), 1))
    return float(np.mean(out))
print("edges inside the two halves: training %.3f | samples %.3f | random %.3f" %
      (inside_fraction(data[:200]), inside_fraction(S), inside_fraction(sym((torch.rand(200, N, N) < d_data).float()))))

### 6. Boltz-2 on a protein and a ligand (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU and `pip install boltz`. Check the Boltz repository README for the current input format and flags.
yaml_text = """
version: 1
sequences:
  - protein:
      id: A
      sequence: <PASTE A PROTEIN SEQUENCE>
  - ligand:
      id: B
      smiles: <PASTE A LIGAND SMILES STRING>
properties:
  - affinity:
      binder: B
"""
open("pair.yaml", "w").write(yaml_text)
# In a notebook cell:  !boltz predict pair.yaml --use_msa_server
# Output: predicted structure files plus a confidence file and an affinity file. Open the structure in a molecular viewer.

### TODO
1. Sample with 5, 10 and 100 steps. How do the statistics change?
2. Remove the `A2` and `A3` inputs from the denoiser and retrain. What is lost, and why?
3. Read the DiGress paper's section on the noise model. How does it differ from the coin-flip noise used here?
4. Close the gap to the training statistics: try a wider denoiser, longer training, or replacing the hand-made inputs with a small message-passing network over node pairs.